# Experiment 3: Logit Lens on Factual Recall — Zamba2-1.2B

**Paper:** Mechanistic Interpretability of Hybrid SSM-Attention Models  
**RQ3:** How do SSM layers and shared attention layers divide representational labor?

## Method: Logit Lens on Factual Recall Prompts

Apply `ln_final + unembed` to each layer's residual stream output on factual recall prompts  
(e.g., "The Eiffel Tower is in" → correct next token: " Paris").

**Key question:** Where does the model "know" the answer?
- If hybrid layers (shared attention) show sharp probability jumps → attention consolidates factual recall
- If Mamba layers show gradual accumulation → SSM builds fact lookup, attention refines
- If layer 23 jumps sharply (as in SSMI) → same layer does both induction and factual recall → general consolidation role

Same logit-lens method as Exp 1 — only the prompts change.

In [ ]:
from google.colab import files
files.download('logit_lens_factual_zamba2_results.json')
files.download('logit_lens_factual_zamba2.png')
files.download('logit_lens_factual_zamba2.pdf')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ── 1. Install ────────────────────────────────────────────────────────────────
!pip install -q git+https://github.com/TransformerLensOrg/TransformerLens.git@dev
!pip install -q transformers>=4.47.0 einops jaxtyping

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 5.4 MB/s eta 0:00:00


In [ ]:
# ── 2. Imports ────────────────────────────────────────────────────────────────
import gc, json
from typing import List, Dict, Tuple
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import torch
from transformer_lens.model_bridge.bridge import TransformerBridge

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if torch.cuda.is_available() else torch.float32
MODEL  = "Zyphra/Zamba2-1.2B"
print(f"Device: {DEVICE} | dtype: {DTYPE}")

Device: cuda | dtype: torch.bfloat16


In [ ]:
# ── 3. Load model ─────────────────────────────────────────────────────────────
bridge = TransformerBridge.boot_transformers(MODEL, device=DEVICE, dtype=DTYPE)
tokenizer = bridge.tokenizer
n_layers = len(bridge.blocks)

# Layer type list — must read from hf_model.config, NOT bridge.cfg
hf_model = bridge.original_model
lbt = list(getattr(hf_model.config, "layers_block_type", []))
hybrid_layers = [i for i, t in enumerate(lbt) if t == "hybrid"]

print(f"Layers: {n_layers} | Mamba: {lbt.count('mamba')} | Hybrid: {lbt.count('hybrid')}")
print(f"Hybrid layer indices: {hybrid_layers}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/1.71k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.86G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/406 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/995 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Layers: 38 | Mamba: 32 | Hybrid: 6
Hybrid layer indices: [5, 11, 17, 23, 29, 35]


/usr/local/lib/python3.12/dist-packages/transformer_lens/model_bridge/bridge.py:488: UserWarning: Hook alias 'hook_mixer_in' -> 'mixer.hook_in' on SSMBlockBridge(name='model.layers.5') did not resolve; this hook will not be accessible.
  getattr(module, "_register_aliases")()
/usr/local/lib/python3.12/dist-packages/transformer_lens/model_bridge/bridge.py:488: UserWarning: Hook alias 'hook_mixer_out' -> 'mixer.hook_out' on SSMBlockBridge(name='model.layers.5') did not resolve; this hook will not be accessible.
  getattr(module, "_register_aliases")()
/usr/local/lib/python3.12/dist-packages/transformer_lens/model_bridge/bridge.py:488: UserWarning: Hook alias 'hook_mixer_in' -> 'mixer.hook_in' on SSMBlockBridge(name='model.layers.11') did not resolve; this hook will not be accessible.
  getattr(module, "_register_aliases")()
/usr/local/lib/python3.12/dist-packages/transformer_lens/model_bridge/bridge.py:488: UserWarning: Hook alias 'hook_mixer_out' -> 'mixer.hook_out' on SSMBlockBridge(na

In [ ]:
# ── 4. Factual recall prompts ──────────────────────────────────────────────────
# Each entry: (prompt, correct_next_token_string)
# Chosen to be clear single-token facts the model should know.
# We verify each tokenizes to a single target token before running.

PROMPTS_RAW = [
    # Geography
    ("The capital of France is", " Paris"),
    ("The capital of Japan is", " Tokyo"),
    ("The capital of Germany is", " Berlin"),
    ("The capital of Italy is", " Rome"),
    ("The capital of Spain is", " Madrid"),
    ("The capital of Australia is", " Canberra"),
    ("The capital of Canada is", " Ottawa"),
    ("The capital of Brazil is", " Bras"),     # Brasília tokenizes differently
    ("The Eiffel Tower is located in", " Paris"),
    ("Mount Everest is located in", " Nepal"),
    # Science facts
    ("The chemical symbol for gold is", " Au"),
    ("The chemical symbol for water is", " H"),
    ("The speed of light is approximately", " 3"),
    ("The atomic number of carbon is", " 6"),
    # History / culture
    ("Shakespeare was born in", " Strat"),
    ("The first US president was George", " Washington"),
    ("The author of 1984 is George", " Or"),
    ("The painter of the Mona Lisa was Leonardo", " da"),
    # Simple language completions
    ("The sun rises in the", " east"),
    ("Water boils at 100 degrees", " Celsius"),
]

# Tokenize and verify each prompt has a single-token target
prompts = []
for text, target_str in PROMPTS_RAW:
    input_ids = tokenizer.encode(text, return_tensors="pt")
    target_tokens = tokenizer.encode(target_str, add_special_tokens=False)
    if len(target_tokens) == 1:
        target_id = target_tokens[0]
        prompts.append({
            "text": text,
            "target_str": target_str.strip(),
            "target_id": target_id,
            "input_ids": input_ids,
            "last_pos": input_ids.shape[1] - 1,
        })
        print(f"  OK  '{text}' -> '{target_str.strip()}' (token {target_id})")
    else:
        print(f"  SKIP '{text}' -> '{target_str}' (multi-token: {target_tokens})")

print(f"\n{len(prompts)} valid single-token prompts ready.")

In [ ]:
# ── 5. GPU warmup ──────────────────────────────────────────────────────────────
print("GPU warmup (Mamba2 CUDA JIT — up to 5 min first time)...")
with torch.no_grad():
    _ = bridge(torch.tensor([[1, 2, 3, 4, 5]]).to(DEVICE))
if torch.cuda.is_available(): torch.cuda.synchronize()
print("Warmup done.")

# Unembedding matrix and final norm
try:
    W_U = bridge.unembed.original_module.weight.detach().float()   # [vocab, d_model]
    ln_final_mod = bridge.ln_final.original_module
except Exception:
    W_U = hf_model.lm_head.weight.detach().float()
    ln_final_mod = hf_model.model.final_layernorm

print(f"W_U shape: {W_U.shape}")

GPU warmup (Mamba2 CUDA JIT — up to 5 min first time)...
Warmup done.
W_U shape: torch.Size([32000, 2048])


In [ ]:
# ── 6. Logit-lens factual recall (1 fwd pass per prompt) ───────────────────────
hook_filter = lambda name: name.endswith("hook_out") and name.startswith("blocks.")

@torch.no_grad()
def logit_lens_factual(bridge, prompts):
    """
    For each prompt, at each layer, compute log-prob of the correct target token
    at the final prompt position (predict next token).
    Returns: (n_prompts, n_layers) array of log-probs.
    """
    n_layers = len(bridge.blocks)
    all_scores = np.zeros((len(prompts), n_layers))

    for p_idx, p in enumerate(prompts):
        print(f"  [{p_idx+1:2d}/{len(prompts)}] '{p['text'][:40]}...' -> '{p['target_str']}'", flush=True)
        input_ids = p["input_ids"].to(DEVICE)
        last_pos  = p["last_pos"]
        target_id = p["target_id"]

        _, cache = bridge.run_with_cache(input_ids, names_filter=hook_filter)

        for layer_idx in range(n_layers):
            key = f"blocks.{layer_idx}.hook_out"
            if key not in cache:
                all_scores[p_idx, layer_idx] = float("nan")
                continue
            residual = cache[key].float()              # [1, seq_len, d_model]
            normed   = ln_final_mod(residual)           # [1, seq_len, d_model]
            logits   = normed[0, last_pos] @ W_U.T      # [vocab]
            lp       = torch.log_softmax(logits, dim=-1)
            all_scores[p_idx, layer_idx] = lp[target_id].item()

        del cache
        if torch.cuda.is_available(): torch.cuda.empty_cache()

    return all_scores

print("Running logit-lens factual recall...")
all_scores = logit_lens_factual(bridge, prompts)
# Mean across prompts (ignore any nans)
mean_scores = np.nanmean(all_scores, axis=0)

print(f"\nDone.")
print(f"Peak logprob: {mean_scores.max():.3f} at layer {mean_scores.argmax()} ({lbt[mean_scores.argmax()]})")
print(f"Mamba mean: {mean_scores[[i for i,t in enumerate(lbt) if t=='mamba']].mean():.3f}")
print(f"Hybrid mean: {mean_scores[[i for i,t in enumerate(lbt) if t=='hybrid']].mean():.3f}")

Running logit-lens factual recall...
  [ 1/13] 'The capital of France is...' -> 'Paris'
  [ 2/13] 'The capital of Japan is...' -> 'Tokyo'
  [ 3/13] 'The capital of Germany is...' -> 'Berlin'
  [ 4/13] 'The capital of Italy is...' -> 'Rome'
  [ 5/13] 'The capital of Spain is...' -> 'Madrid'
  [ 6/13] 'The capital of Brazil is...' -> 'Bras'
  [ 7/13] 'The Eiffel Tower is located in...' -> 'Paris'
  [ 8/13] 'The chemical symbol for gold is...' -> 'Au'
  [ 9/13] 'The chemical symbol for water is...' -> 'H'
  [10/13] 'The first US president was George...' -> 'Washington'
  [11/13] 'The author of 1984 is George...' -> 'Or'
  [12/13] 'The painter of the Mona Lisa was Leonard...' -> 'da'
  [13/13] 'The sun rises in the...' -> 'east'

Done.
Peak logprob: -0.043 at layer 35 (hybrid)
Mamba mean: -10.265
Hybrid mean: -7.666


In [ ]:
# ── 7. Per-prompt inspection ───────────────────────────────────────────────────
print("\nPer-prompt peak layer:")
for i, p in enumerate(prompts):
    peak_layer = int(np.argmax(all_scores[i]))
    peak_score = all_scores[i, peak_layer]
    peak_type  = lbt[peak_layer]
    print(f"  '{p['text'][:35]:35s}' -> '{p['target_str']:12s}' | peak=L{peak_layer:2d} ({peak_type:6s}) logp={peak_score:.2f}")

In [ ]:
# ── 8. Main plot: mean log-prob per layer ──────────────────────────────────────
colors = ["#e05c5c" if t == "hybrid" else "#5c8de0" for t in lbt]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(np.arange(n_layers), mean_scores, color=colors, width=0.8, alpha=0.85)
for pos in hybrid_layers:
    ax.axvline(x=pos, color="#cc3333", linewidth=0.8, linestyle="--", alpha=0.5)

ax.legend(handles=[
    Patch(facecolor="#5c8de0", label="Mamba-2 (SSM) layer"),
    Patch(facecolor="#e05c5c", label="Hybrid layer (SSM + shared attention)"),
], loc="lower right", fontsize=10)
ax.set_xlabel("Layer index", fontsize=12)
ax.set_ylabel("Mean log-prob of correct token (logit lens)", fontsize=11)
ax.set_title("Logit Lens: Factual Recall Accuracy per Layer — Zamba2-1.2B", fontsize=13)
ax.axhline(y=0, color="black", linewidth=0.8)
ax.set_xlim(-0.5, n_layers - 0.5)

plt.tight_layout()
plt.savefig("logit_lens_factual_zamba2.pdf", bbox_inches="tight", dpi=150)
plt.savefig("logit_lens_factual_zamba2.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: logit_lens_factual_zamba2.pdf + .png")

In [ ]:
# ── 9. Comparison plot: SSMI vs factual recall per layer ──────────────────────
# Load Exp 1 SSMI results to overlay
try:
    from google.colab import drive
    ssmi_path = "ssmi_zamba2_results.json"   # adjust if needed
except Exception:
    ssmi_path = "ssmi_zamba2_results.json"

try:
    with open(ssmi_path) as f:
        ssmi_data = json.load(f)
    ssmi_scores = np.array(ssmi_data["ssmi_scores"])
    has_ssmi = True
    print("Loaded SSMI data for comparison plot.")
except FileNotFoundError:
    has_ssmi = False
    print("SSMI file not found — skipping comparison plot. Upload ssmi_zamba2_results.json to Colab if you want it.")

if has_ssmi:
    fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
    fig.suptitle("Zamba2-1.2B: Induction vs Factual Recall — Logit Lens Comparison", fontsize=13)

    for ax, scores, ylabel, title in [
        (axes[0], ssmi_scores, "SSMI (log-prob, induction)", "Exp 1: SSM Induction Score"),
        (axes[1], mean_scores, "Mean log-prob (factual recall)", "Exp 3: Factual Recall"),
    ]:
        ax.bar(np.arange(n_layers), scores, color=colors, width=0.8, alpha=0.85)
        for pos in hybrid_layers:
            ax.axvline(x=pos, color="#cc3333", linewidth=0.8, linestyle="--", alpha=0.5)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.set_title(title, fontsize=11)
        ax.axhline(y=0, color="black", linewidth=0.8)
        ax.set_xlim(-0.5, n_layers - 0.5)

    axes[1].set_xlabel("Layer index", fontsize=12)
    axes[1].legend(handles=[
        Patch(facecolor="#5c8de0", label="Mamba-2 (SSM)"),
        Patch(facecolor="#e05c5c", label="Hybrid (SSM + shared attn)"),
    ], loc="lower right", fontsize=9)

    plt.tight_layout()
    plt.savefig("logit_lens_comparison_zamba2.pdf", bbox_inches="tight", dpi=150)
    plt.savefig("logit_lens_comparison_zamba2.png", bbox_inches="tight", dpi=150)
    plt.show()
    print("Saved: logit_lens_comparison_zamba2.pdf + .png")

In [ ]:
# ── 10. Save results ───────────────────────────────────────────────────────────
import json

mamba_idx  = [i for i, t in enumerate(lbt) if t == "mamba"]
hybrid_idx = [i for i, t in enumerate(lbt) if t == "hybrid"]
peak_layer = int(mean_scores.argmax())

results = {
    "model": MODEL,
    "method": "logit_lens_factual_recall",
    "n_prompts": len(prompts),
    "prompts": [{"text": p["text"], "target": p["target_str"]} for p in prompts],
    "layer_types": lbt,
    "mean_logprob_per_layer": mean_scores.tolist(),
    "all_logprob_scores": all_scores.tolist(),
    "summary": {
        "peak_layer": peak_layer,
        "peak_layer_type": lbt[peak_layer],
        "peak_logprob": float(mean_scores[peak_layer]),
        "mamba_mean": float(mean_scores[mamba_idx].mean()),
        "hybrid_mean": float(mean_scores[hybrid_idx].mean()),
        "hybrid_layer_scores": {str(i): float(mean_scores[i]) for i in hybrid_idx},
    }
}

with open("logit_lens_factual_zamba2_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved: logit_lens_factual_zamba2_results.json")
print(f"\nSummary:")
print(f"  Peak layer: {peak_layer} ({lbt[peak_layer]}) logp={mean_scores[peak_layer]:.3f}")
print(f"  Mamba mean: {mean_scores[mamba_idx].mean():.3f}")
print(f"  Hybrid mean: {mean_scores[hybrid_idx].mean():.3f}")
print(f"  Per-hybrid-layer: { {i: round(mean_scores[i],3) for i in hybrid_idx} }")

Saved: logit_lens_factual_zamba2_results.json

Summary:
  Peak layer: 35 (hybrid) logp=-0.043
  Mamba mean: -10.265
  Hybrid mean: -7.666
  Per-hybrid-layer: {5: np.float64(-14.334), 11: np.float64(-13.237), 17: np.float64(-11.54), 23: np.float64(-4.994), 29: np.float64(-1.845), 35: np.float64(-0.043)}


## What to look for

**If SSMI and factual recall both peak at layer 23:**
→ Layer 23's hybrid attention is a **general-purpose consolidation layer** — it doesn't implement a specific pattern (copy score ~0), but it sharply improves both induction and factual recall. This is the most interesting finding: weight-tied attention at position 23 acts as a knowledge integrator, not a pattern matcher.

**If factual recall peaks at a different layer than SSMI:**
→ Induction and factual recall use different circuits. Layer 23's SSMI jump is specific to sequence copying, not a global consolidation effect.

**If Mamba layers show gradual monotonic improvement for factual recall:**
→ SSM layers build fact lookup through hidden state compression; hybrid attention provides refinement.

Either outcome is publishable and directly answers RQ3.